# Model Training
Trains a demonstration Random Forest pipeline on synthetic data.

In [ ]:
from pathlib import Path
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
root = Path.cwd().parent if Path.cwd().name == "Notebooks" else Path.cwd()
df = pd.read_csv(root / "data/cleaned_data/used_cars_cleaned.csv")
X = df.drop(columns="selling_price")
y = df["selling_price"]

In [ ]:
categorical = ["brand", "seller_type", "fuel_type", "transmission_type"]
numeric = [c for c in X.columns if c not in categorical]
preprocess = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), categorical), ("num", "passthrough", numeric)])
model = Pipeline([("preprocessor", preprocess), ("regressor", RandomForestRegressor(n_estimators=90, random_state=42, n_jobs=-1))])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model.fit(X_train, y_train)
pred = model.predict(X_test)
print("MAE:", mean_absolute_error(y_test, pred))
print("R2:", r2_score(y_test, pred))

In [ ]:
joblib.dump(model, root / "models/car_price_pipeline.pkl")